# Projeto Final - 02. Bronze para Silver (Delta)
Atende diretamente:
- **RF02**: calcula taxa de refugo e produtividade por hora, por ordem
- **RF07**: garante idempotencia (reprocessar o mesmo dia nao duplica)
- **RNF02**: dado de producao consolidado e confiavel para o dia seguinte
Fica a nivel de ordem (uma linha por ordem de producao) -- a agregacao por
linha/maquina/dia fica pra Gold.

In [11]:
# Vazio = processa toda a Bronze acumulada
data_referencia = ""

StatementMeta(industry, 8, 11, Finished, Available, Finished, False)

In [12]:
caminho_bronze = "abfss://inicial-datalake@dlcursoazure.dfs.core.windows.net/BRONZE/ordens_producao_delta/"
caminho_silver = "abfss://inicial-datalake@dlcursoazure.dfs.core.windows.net/SILVER/ordens_producao_delta/"
df_bronze = spark.read.format("delta").load(caminho_bronze)
if data_referencia != "":
    df_bronze = df_bronze.filter(df_bronze.data_producao == data_referencia)
print("Linhas lidas da bronze:", df_bronze.count())

StatementMeta(industry, 8, 12, Finished, Available, Finished, False)

Linhas lidas da bronze: 168


## 1. Qualidade de dado: duracao_minutos = 0
O CSV tem ordens com `duracao_minutos = 0` (erro de apontamento na origem). Isso
quebraria a formula de produtividade (divisao por zero). Em vez de descartar a ordem-- ela ainda tem informacao valida de produzido/refugado -- sinalizamos com uma flag
e usamos `NULL` na produtividade apenas para essas linhas, mantendo o resto intacto
(RF02 nao pode ser calculado errado, mas os outros requisitos, como taxa de refugo,
continuam validos para essa ordem).

In [13]:
from pyspark.sql.functions import col, when, trim, upper, to_date, round as spark_round
df_limpo = (
    df_bronze
    .withColumn("linha_producao", upper(trim(col("linha_producao"))))
    .withColumn("id_maquina", upper(trim(col("id_maquina"))))
    .withColumn("turno", trim(col("turno")))
    .withColumn("data_producao", to_date(col("data_producao"), "yyyy-MM-dd"))
    .withColumn("flag_duracao_invalida", col("duracao_minutos") == 0)
    .filter(col("quantidade_produzida") > 0)  # ordem sem producao nao faz sentido de negocio
)

StatementMeta(industry, 8, 13, Finished, Available, Finished, False)

## 2. RF02 -- Formulas de negocio

In [14]:
from pyspark.sql.functions import col, when, trim, upper, to_date, round as spark_round
df_limpo = (
    df_bronze
    .withColumn("linha_producao", upper(trim(col("linha_producao"))))
    .withColumn("id_maquina", upper(trim(col("id_maquina"))))
    .withColumn("turno", trim(col("turno")))
    .withColumn("data_producao", to_date(col("data_producao"), "yyyy-MM-dd"))
    .withColumn("flag_duracao_invalida", col("duracao_minutos") == 0)
    .filter(col("quantidade_produzida") > 0)  # ordem sem producao nao faz sentido de negocio
)

StatementMeta(industry, 8, 14, Finished, Available, Finished, False)

In [15]:
df_metricas = (
    df_limpo
    .withColumn(
        "taxa_refugo_pct",
        spark_round((col("quantidade_refugada") / col("quantidade_produzida")) * 100, 2)
    )
    .withColumn(
        "produtividade_hora",
        when(
            col("flag_duracao_invalida"),
            None  # nao calculavel -- evita divisao por zero, mantem a ordem
        ).otherwise(
            spark_round(col("quantidade_produzida") / (col("duracao_minutos") / 60), 2)
        )
    )
)
df_metricas.select(
    "id_ordem", "linha_producao", "id_maquina", "quantidade_produzida",
    "quantidade_refugada", "taxa_refugo_pct", "produtividade_hora", "flag_duracao_invalida"
).show(10)

StatementMeta(industry, 8, 15, Finished, Available, Finished, False)

+--------+--------------+----------+--------------------+-------------------+---------------+------------------+---------------------+
|id_ordem|linha_producao|id_maquina|quantidade_produzida|quantidade_refugada|taxa_refugo_pct|produtividade_hora|flag_duracao_invalida|
+--------+--------------+----------+--------------------+-------------------+---------------+------------------+---------------------+
|       1|            L1|       M01|                 468|                 12|           2.56|             64.11|                false|
|       2|            L1|       M01|                 521|                 15|           2.88|             69.16|                false|
|       3|            L1|       M01|                 840|                  8|           0.95|            113.77|                false|
|       4|            L1|       M02|                 738|                 30|           4.07|            102.03|                false|
|       5|            L1|       M02|                 34

## 3. RF07 -- Idempotencia (dedup por chave de negocio)
`id_ordem` e a chave unica de uma ordem de producao. Reprocessar o mesmo arquivo
(ex: pipeline rodado duas vezes por engano) nao pode gerar linha duplicada.

In [16]:
total_antes = df_metricas.count()
df_silver_final = df_metricas.dropDuplicates(["id_ordem"])
total_depois = df_silver_final.count()
print(f"Antes do dedup: {total_antes} | Depois: {total_depois} | Duplicatas removidas: {total_antes - total_depois}")

StatementMeta(industry, 8, 16, Finished, Available, Finished, False)

Antes do dedup: 168 | Depois: 168 | Duplicatas removidas: 0


## 4. Gravando a Silver (mesma estrategia de particionamento da Bronze)

In [17]:

(
    df_silver_final.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("linha_producao")
    .save(caminho_silver)
)    
print("Silver gravada:", caminho_silver)
print("Total de linhas:", df_silver_final.count())
print("Ordens com duracao invalida (produtividade nula):", df_silver_final.filter(col("flag_duracao_invalida")).count())


StatementMeta(industry, 8, 17, Finished, Available, Finished, False)

Silver gravada: abfss://inicial-datalake@dlcursoazure.dfs.core.windows.net/SILVER/ordens_producao_delta/
Total de linhas: 168
Ordens com duracao invalida (produtividade nula): 2


In [18]:
mssparkutils.notebook.exit(f"OK - {df_silver_final.count()} linhas gravadas na silver")

StatementMeta(industry, 8, 18, Finished, Available, Finished, False)

ExitValue: OK - 168 linhas gravadas na silver